In [1]:
import os
import json
import pickle
from tqdm import tqdm

# =========================
# 0) 固定调用关系（与你给的脚本一致）
# =========================
DEFAULT_CALL_RELATIONS = [
    "frontend -> adservice",
    "frontend -> cartservice",
    "frontend -> checkoutservice",
    "frontend -> currencyservice",
    "frontend -> recommendationservice",
    "frontend -> productcatalogservice",
    "frontend -> shippingservice",
    "checkoutservice -> cartservice",
    "checkoutservice -> currencyservice",
    "checkoutservice -> emailservice",
    "checkoutservice -> paymentservice",
    "checkoutservice -> productcatalogservice",
    "checkoutservice -> shippingservice",
    "recommendationservice -> productcatalogservice",
]

DEFAULT_POD_SUFFIXES = ["-0", "-1", "-2", "2-0"]


def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)
    return p


# =========================
# 1) 构图：返回 edge_index = [src_list, dst_list]
# =========================
def build_aiops_topology_edge_index(
    all_entity_list: list,
    service_order_list: list,
    call_relations=DEFAULT_CALL_RELATIONS,
    pod_suffixes=DEFAULT_POD_SUFFIXES,
    add_self_loops: bool = True,
):
    name2idx = {name: i for i, name in enumerate(all_entity_list)}
    src, dst = [], []

    # 1) service <-> pod
    for service in service_order_list:
        if service not in name2idx:
            raise ValueError(f"[topology] service not in all_entity_list: {service}")
        s_idx = name2idx[service]

        for suf in pod_suffixes:
            pod = f"{service}{suf}"
            if pod not in name2idx:
                raise ValueError(f"[topology] pod not in all_entity_list: {pod}")

            p_idx = name2idx[pod]
            src.append(p_idx); dst.append(s_idx)  # pod -> service
            src.append(s_idx); dst.append(p_idx)  # service -> pod

    # 2) service <-> service call graph（双向）
    for rel in call_relations:
        caller, callee = rel.split(" -> ")
        if caller not in name2idx or callee not in name2idx:
            raise ValueError(f"[topology] caller/callee missing in all_entity_list: {rel}")

        c1, c2 = name2idx[caller], name2idx[callee]
        src.append(c1); dst.append(c2)
        src.append(c2); dst.append(c1)

    # 3) self loops
    if add_self_loops:
        for i in range(len(all_entity_list)):
            src.append(i); dst.append(i)

    return [src, dst]


# =========================
# 2) 枚举 AIOps cases：dataset_type/date/cloudbed
# =========================
def iter_aiops_cases(data_base: str):
    for dataset_type in sorted(os.listdir(data_base)):
        ds_dir = os.path.join(data_base, dataset_type)
        if not os.path.isdir(ds_dir):
            continue

        for date in sorted(os.listdir(ds_dir)):
            date_dir = os.path.join(ds_dir, date)
            if not os.path.isdir(date_dir):
                continue

            for cloudbed in sorted(os.listdir(date_dir)):
                cb_dir = os.path.join(date_dir, cloudbed)
                if not os.path.isdir(cb_dir):
                    continue
                yield dataset_type, date, cloudbed


# =========================
# 3) step A：写 raw_data 下的 edge_index.pkl（与你脚本一致）
# =========================
def generate_aiops_topology_files(
    data_base: str,
    out_base: str,
    all_entity_list: list,
    service_order_list: list,
    skip_bad_case: bool = True,
):
    edge_index = build_aiops_topology_edge_index(
        all_entity_list=all_entity_list,
        service_order_list=service_order_list,
    )

    for dataset_type, date, cloudbed in iter_aiops_cases(data_base):
        if skip_bad_case and date == "2022-03-24" and cloudbed in ["cloudbed-1", "cloudbed-2"]:
            continue

        out_dir = ensure_dir(os.path.join(out_base, dataset_type, date, cloudbed, "resource_entity"))
        out_pkl = os.path.join(out_dir, "edge_index.pkl")
        with open(out_pkl, "wb") as f:
            pickle.dump(edge_index, f)

        print(f"[OK] saved topology -> {out_pkl}")


# =========================
# 4) step B：把 edge_index 按 window_size 对齐成 dataset 级别 list
#    输出：dataset/ent_edge_index/ent_edge_index_window_size_{w}.pkl
# =========================
def load_time_intervals_if_exists(analysis_dir: str, window_size: int):
    """
    支持你 GAIA/SN/TT 的做法：如果有预先保存的 time_intervals_window_{w}.json 就优先读
    期望结构示例：
      {
        "train_valid": [[date, cloudbed, st, et], ...],
        "test":        [[date, cloudbed, st, et], ...]
      }
    """
    p = os.path.join(analysis_dir, f"time_intervals_window_{window_size}.json")
    if not os.path.exists(p):
        return None
    with open(p, "r", encoding="utf-8") as f:
        obj = json.load(f)
    out = {}
    for k in ["train_valid", "test"]:
        out[k] = [(x[0], x[1], int(x[2]), int(x[3])) for x in obj.get(k, [])]
    return out


def fallback_intervals_from_timestamps(raw_log_csv_path: str, window_size_min: int):
    """
    fallback：如果没有 TimeIntervalLabelGenerator / interval json
    就用 raw_log 的 timestamp 轴做滑窗：
    - window_size_min：以“分钟”为单位（AIOps 原始是 60s 粒度）
    return: [(st, et), ...]  st/et 为秒
    """
    import pandas as pd
    df = pd.read_csv(raw_log_csv_path, usecols=["timestamp"])
    ts = df["timestamp"].astype(int).values
    out = []
    W = window_size_min
    for i in range(0, len(ts) - W):
        st = int(ts[i])
        et = int(ts[i + W])  # 半开区间
        out.append((st, et))
    return out


def build_cases_index(raw_data_root: str):
    """
    用 raw_data_root 枚举 case，并找到 raw_log 的 timestamp 轴：
      raw_data_root/<dataset_type>/<date>/<cloudbed>/raw_log/log_patterns_count.csv
    """
    case_map = {}
    for dataset_type in sorted(os.listdir(raw_data_root)):
        ds_dir = os.path.join(raw_data_root, dataset_type)
        if not os.path.isdir(ds_dir):
            continue
        for date in sorted(os.listdir(ds_dir)):
            date_dir = os.path.join(ds_dir, date)
            if not os.path.isdir(date_dir):
                continue
            for cloudbed in sorted(os.listdir(date_dir)):
                cb_dir = os.path.join(date_dir, cloudbed)
                raw_log_csv = os.path.join(cb_dir, "raw_log", "log_patterns_count.csv")
                edge_pkl = os.path.join(cb_dir, "resource_entity", "edge_index.pkl")
                if os.path.exists(raw_log_csv) and os.path.exists(edge_pkl):
                    case_map.setdefault(dataset_type, {}).setdefault(date, {})[cloudbed] = {
                        "raw_log_csv": raw_log_csv,
                        "edge_pkl": edge_pkl,
                    }
    return case_map


def load_time_intervals_from_label_pkl(temp_data_storage: str, window_size: int):
    """
    读取:
      {temp_data_storage}/dataset/time_interval_and_label/time_interval_window_size_{w}.pkl
    返回:
      {
        "train_valid": [(date, cloudbed, st, et), ...],
        "test": [(date, cloudbed, st, et), ...]
      }
    """
    p = os.path.join(
        temp_data_storage,
        "dataset",
        "time_interval_and_label",
        f"time_interval_window_size_{window_size}.pkl",
    )
    if not os.path.exists(p):
        raise FileNotFoundError(f"time interval pkl not found: {p}")

    with open(p, "rb") as f:
        obj = pickle.load(f)

    out = {}
    for split in ["train_valid", "test"]:
        out[split] = [(x[0], x[1], int(x[2]), int(x[3])) for x in obj["time_interval"][split]]
    return out


def generate_ent_edge_index_dataset(
    raw_data_root,
    temp_data_storage,
    dataset_out_dir,
    window_size_list,
):
    """
    raw_data_root:  ../../temp_data/aiops22/raw_data
    analysis_dir:   ../../temp_data/aiops22/analysis/ent_edge_index 或 analysis/log（你放 interval json 的地方）
    dataset_out_dir:../../temp_data/aiops22/dataset/ent_edge_index

    split_dataset_type_map：把 split 映射到 dataset_type
      默认 AIOps：
        train_valid -> "training_data_with_faults"
        test        -> "test_data"
    """

    split_dataset_type_map = {
        "train_valid": "training_data_with_faults",
        "test": "test_data",
    }

    ensure_dir(dataset_out_dir)
    case_map = build_cases_index(raw_data_root)

    for window_size in window_size_list:
        # ✅ 唯一时间窗来源：label generator 生成的 pkl
        intervals = load_time_intervals_from_label_pkl(temp_data_storage, window_size)

        relation_dict = {"train_valid": [], "test": []}

        for split in ["train_valid", "test"]:
            ds_type = split_dataset_type_map[split]
            if ds_type not in case_map:
                continue

            # intervals: [(date, cloudbed, st, et), ...]
            split_intervals = intervals.get(split, [])
            for (date, cloudbed, _st, _et) in tqdm(split_intervals, desc=f"edge_index {split} w={window_size}"):
                info = case_map[ds_type].get(date, {}).get(cloudbed)
                if info is None:
                    continue
                with open(info["edge_pkl"], "rb") as f:
                    edge_index = pickle.load(f)
                relation_dict[split].append(edge_index)


        out_pkl = os.path.join(dataset_out_dir, f"ent_edge_index_window_size_{window_size}.pkl")
        '''
            {
              "train_valid": [
                  edge_index_window1,
                  edge_index_window2,
                  ...
              ],
              "test": [
                  edge_index_window1,
                  ...
              ]
            }
            edge_index_window1 = [src, dst]
        '''
        with open(out_pkl, "wb") as f:
            pickle.dump(relation_dict, f)

        print(f"[OK] saved -> {out_pkl}")


# =========================
# 5) 一个“Generator 类”封装（对齐你工程写法）
# =========================
class EntEdgeIndexGenerator:
    """
    轻量版：不依赖 BaseGenerator/Dao
    - generate_raw_topology_files(): 写 resource_entity/edge_index.pkl
    - generate_windowed_ent_edge_index(): 写 dataset/ent_edge_index/ent_edge_index_window_size_{w}.pkl
    """
    def __init__(
        self,
        data_base: str,
        temp_data_storage: str,
        node_list: list,
        service_list: list,
        pod_list: list,
        service_order_list: list,
        window_size_list: list,
        skip_bad_case: bool = True,
        analysis_interval_dir: str | None = None,
    ):
        self.data_base = data_base
        self.temp_data_storage = temp_data_storage
        self.raw_data_root = os.path.join(temp_data_storage, "raw_data")
        self.dataset_out_dir = os.path.join(temp_data_storage, "dataset", "ent_edge_index")
        self.analysis_dir = analysis_interval_dir or os.path.join(temp_data_storage, "analysis", "log")

        self.node_list = node_list
        self.service_list = service_list
        self.pod_list = pod_list
        self.service_order_list = service_order_list

        self.window_size_list = window_size_list
        self.skip_bad_case = skip_bad_case

    def generate_raw_topology_files(self):
        generate_aiops_topology_files_v2(
            data_base=self.data_base,
            out_base=self.raw_data_root,
            node_list=self.node_list,
            service_list=self.service_list,
            pod_list=self.pod_list,
            service_order_list=self.service_order_list,
            skip_bad_case=self.skip_bad_case,
        )

    def generate_windowed_ent_edge_index(self):
        generate_ent_edge_index_dataset(
            raw_data_root=self.raw_data_root,
            temp_data_storage=self.temp_data_storage,
            dataset_out_dir=self.dataset_out_dir,
            window_size_list=self.window_size_list,
        )


def build_aiops_topology_edge_index_v2(
    node_list: list,
    service_list: list,
    pod_list: list,
    call_relations=DEFAULT_CALL_RELATIONS,
    add_self_loops: bool = True,
):
    """
    节点顺序必须和你最终 meta_data['ent_names'] 一致：
      all_entity_list = node_list + service_list + pod_list
    """
    all_entity_list = node_list + service_list + pod_list
    name2idx = {name: i for i, name in enumerate(all_entity_list)}
    src, dst = [], []

    # 1) service <-> pod
    # 规则：pod 名类似 "adservice-1" 这种，用 rsplit('-',1) 找到 service
    for pod in pod_list:
        if pod not in name2idx:
            raise ValueError(f"[topology] pod missing in all_entity_list: {pod}")
        p_idx = name2idx[pod]

        # 从 pod 推 service 名（与你 AIOps pod 命名一致时适用：xxx-1 / xxx-2）
        svc = pod.rsplit("-", 1)[0]          # 先取 adservice2
        if svc not in name2idx:
            # 兼容 xxx2-0 / xxx2-1 / xxx2-2 这类：把末尾的 "2" 当成第二套部署标记
            if svc.endswith("2") and svc[:-1] in name2idx:
                svc = svc[:-1]
        
        if svc not in name2idx:
            raise ValueError(f"[topology] cannot resolve service for pod={pod}, got svc={svc}")
            
        s_idx = name2idx[svc]

        src.append(p_idx); dst.append(s_idx)  # pod -> service
        src.append(s_idx); dst.append(p_idx)  # service -> pod

    # 2) service <-> service call graph（双向）
    for rel in call_relations:
        caller, callee = rel.split(" -> ")
        if caller not in name2idx or callee not in name2idx:
            raise ValueError(f"[topology] caller/callee missing: {rel}")
        c1, c2 = name2idx[caller], name2idx[callee]
        src.append(c1); dst.append(c2)
        src.append(c2); dst.append(c1)

    # 3) self loops
    if add_self_loops:
        for i in range(len(all_entity_list)):
            src.append(i); dst.append(i)

    '''
        edge_index = [
            src,   # 所有边的起点
            dst    # 所有边的终点
        ]

        src = [s1, s2, s3, ...]
        dst = [d1, d2, d3, ...]
    '''
    return [src, dst], all_entity_list


def generate_aiops_topology_files_v2(
    data_base: str,
    out_base: str,
    node_list: list,
    service_list: list,
    pod_list: list,
    service_order_list: list,
    skip_bad_case: bool = True,
):
    edge_index, all_entity_list = build_aiops_topology_edge_index_v2(
        node_list=node_list,
        service_list=service_list,
        pod_list=pod_list,
        call_relations=DEFAULT_CALL_RELATIONS,
        add_self_loops=True
    )

    # sanity check：必须对上 meta 的 E
    E = len(all_entity_list)
    max_id = max(max(edge_index[0]), max(edge_index[1]))
    if max_id != E - 1:
        raise ValueError(f"[topology] id range mismatch: E={E} but max_id={max_id}")

    for dataset_type, date, cloudbed in iter_aiops_cases(data_base):
        if skip_bad_case and date == "2022-03-24" and cloudbed in ["cloudbed-1", "cloudbed-2"]:
            continue

        out_dir = ensure_dir(os.path.join(out_base, dataset_type, date, cloudbed, "resource_entity"))
        out_pkl = os.path.join(out_dir, "edge_index.pkl")
        '''
            edge_index = [
                src,   # 所有边的起点
                dst    # 所有边的终点
            ]
    
            src = [s1, s2, s3, ...]
            dst = [d1, d2, d3, ...]
        '''
        with open(out_pkl, "wb") as f:
            pickle.dump(edge_index, f)

        # 可选：把实体顺序也存一份，后面排查非常好用
        with open(os.path.join(out_dir, "entity_order.json"), "w", encoding="utf-8") as f:
            json.dump(all_entity_list, f, indent=2, ensure_ascii=False)

        print(f"[OK] saved topology -> {out_pkl}  (E={E})")

        
# =========================
# 6) 用法示例（按你给的参数）
# =========================
if __name__ == "__main__":
    node_list = ["node-1","node-2","node-3","node-4","node-5","node-6"]

    service_order_list = [
        "adservice","cartservice","checkoutservice","currencyservice","emailservice",
        "frontend","paymentservice","productcatalogservice","recommendationservice","shippingservice"
    ]

    pod_list = [
        'adservice-0', 
        'adservice-1', 
        'adservice-2', 
        'adservice2-0', 
        'cartservice-0', 
        'cartservice-1', 
        'cartservice-2', 
        'cartservice2-0', 
        'checkoutservice-0', 
        'checkoutservice-1', 
        'checkoutservice-2', 
        'checkoutservice2-0', 
        'currencyservice-0', 
        'currencyservice-1', 
        'currencyservice-2', 
        'currencyservice2-0', 
        'emailservice-0', 
        'emailservice-1', 
        'emailservice-2', 
        'emailservice2-0', 
        'frontend-0', 
        'frontend-1', 
        'frontend-2', 
        'frontend2-0', 
        'paymentservice-0', 
        'paymentservice-1', 
        'paymentservice-2', 
        'paymentservice2-0', 
        'productcatalogservice-0', 
        'productcatalogservice-1', 
        'productcatalogservice-2', 
        'productcatalogservice2-0', 
        'recommendationservice-0', 
        'recommendationservice-1', 
        'recommendationservice-2', 
        'recommendationservice2-0', 
        'shippingservice-0', 
        'shippingservice-1', 
        'shippingservice-2', 
        'shippingservice2-0'
    ]

    gen = EntEdgeIndexGenerator(
        data_base="/home/ZZData/Eadro/preprocess/datasets/aiops2022/",
        temp_data_storage="../../temp_data/aiops22",
        node_list=node_list,
        service_list=service_order_list,
        pod_list=pod_list,
        service_order_list=service_order_list,
        window_size_list=[5, 10, 15],
        skip_bad_case=True,
    )

    gen.generate_raw_topology_files()        # ✅ 会写 edge_index.pkl
    gen.generate_windowed_ent_edge_index()   # ✅ 会写 ent_edge_index_window_size_{w}.pkl

edge_index test w=15: 100%|██████████| 241/241 [00:00<00:00, 112954.21it/s]

[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/2022-05-01/cloudbed/resource_entity/edge_index.pkl  (E=56)
[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/2022-05-03/cloudbed/resource_entity/edge_index.pkl  (E=56)
[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/2022-05-05/cloudbed/resource_entity/edge_index.pkl  (E=56)
[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/2022-05-07/cloudbed/resource_entity/edge_index.pkl  (E=56)
[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/2022-05-09/cloudbed/resource_entity/edge_index.pkl  (E=56)
[OK] saved topology -> ../../temp_data/aiops22/raw_data/test_data/groundtruth/.ipynb_checkpoints/resource_entity/edge_index.pkl  (E=56)
[OK] saved topology -> ../../temp_data/aiops22/raw_data/training_data_normal/2022-03-19/.ipynb_checkpoints/resource_entity/edge_index.pkl  (E=56)
[OK] saved topology -> ../../temp_data/aiops22/raw_data/training_data_normal/2022-03-19/cloud